## Workspace setup

In [ ]:
from datetime import datetime
import tensorflow as tf
import numpy as np
import importlib
from termcolor import colored
import tensorflow_datasets as tfds

# Notebooks run from VSCode use home directory as a base path
# while notebooks run from JupyterLab use the current directory as a base path
import sys, os

home = os.getenv("HOME")
if home == "/home/jovyan":
    sys.path.append("/home/jovyan/ELITPC/TPCReco/PythonAnalysis/python/")
else:
    sys.path.append("/home/akalinow/scratch/ELITPC/TPCReco/PythonAnalysis/python/")
    os.chdir('/home/akalinow/scratch/ELITPC/PythonAnalysis/')

import mltree_tfrecord as mt

### Convert simulated data.

The ROOT files are written by `tpcWriter` with `--output.mlTree true` (see [README](../README.md)).
Their `MLData` tree contains, for each event, the hits and the generated (`sim_*`) and
reconstructed (`reco_*`) track segments. `mt.convert_file` writes it to GZIP compressed
TFRecord files (one `tf.train.Example` per event, charge not normalised), the same as the
command line `python3 mltree_tfrecord.py <ROOT files> -o <output directory>`.
This step does not need TensorFlow and has to be executed only once per dataset.

In [ ]:
%%time
importlib.reload(mt)

#dataPath = '/scratch_cmsse/akalinow/ELITPC/data/PythonAnalysis/TwoProng/'
dataPath = '/home/jovyan/ELITPC/TPCReco/build/resources/'

rootFiles = [dataPath+'SimEvent_Track3D_TwoProng_gun_MC_100k.root']

# Convert ROOT files to TFRecord files in the output directory
outputDir = 'TFRecord_TwoProng_gun_MC_100k'
for rootFile in rootFiles:
    nEvents, shards = mt.convert_file(rootFile, outputDir, eventsPerShard=1000)
    print(rootFile, nEvents, "events in", len(shards), "TFRecord files")

### Load SimEvent and RecoEvent data

Each TFRecord event holds both the generated and the reconstructed segments, so no merging
of separate datasets is needed. `mt.load_merged_dataset` gives the structure of the old merged
datasets: `{'sim': (image, target), 'reco': (image, target)}` with the image normalised to its
maximum and the `(3, 3)` target (rows x, y, z; columns vertex, alpha end, carbon end).
Selections are applied when the dataset is loaded.
`mt.load_dataset` gives other layouts (e.g. three prong events, raw charge).

In [ ]:
%%time
mergedDataset = mt.load_merged_dataset(outputDir+'/*.tfrecord.gz')

# optional selection on the simulated alpha track length [mm]
#minAlphaLength = 10.0
#maxAlphaLength = 10000.0
#mergedDataset = mergedDataset.filter(
#    lambda x: (mt.segment_length(x['sim'][1], 0) > minAlphaLength) &
#              (mt.segment_length(x['sim'][1], 0) < maxAlphaLength)
#)
print(colored("Number of events: ", "blue"), sum(1 for _ in mergedDataset))

### Create a dataset and run benchmark

In [ ]:
import plotting_functions as plf
importlib.reload(plf)

batchSize = 32
dataset = mt.load_merged_dataset(outputDir+'/*.tfrecord.gz', parallel_reads=True)
dataset = dataset.batch(batchSize, drop_remainder=True)
dataset = dataset.take(500).cache().prefetch(tf.data.AUTOTUNE)

print(colored("Timing before caching.","blue"))
tfds.benchmark(dataset,batch_size=batchSize)
print(colored("Timing after caching.", "blue"))
tfds.benchmark(dataset,batch_size=batchSize)

### Load and convert real data events

Real data events have only reconstructed segments. `mt.load_merged_dataset` puts the
reconstructed target also into `'sim'`, to keep the same structure as for simulated data.

In [ ]:
%%time
importlib.reload(mt)

dataPath = '/home/akalinow/scratch/ELITPC/TPCReco/build/resources/'
rootFiles = [dataPath+'RecoEvent_TwoProng_2022-04-12T08-03-44.root']
outputDir = 'TFRecord_TwoProng_2022-04-12T08-03-44'

# Convert ROOT files to TFRecord files in the output directory
for rootFile in rootFiles:
    nEvents, shards = mt.convert_file(rootFile, outputDir)
    print(rootFile, nEvents, "events in", len(shards), "TFRecord files")

mergedDataset = mt.load_merged_dataset(outputDir+'/*.tfrecord.gz')

# select events with the alpha track length in [minAlphaLength, maxAlphaLength] mm
minAlphaLength = 20.0
maxAlphaLength = 100.0
mergedDataset = mergedDataset.filter(
    lambda x: (mt.segment_length(x['sim'][1], 0) > minAlphaLength) &
              (mt.segment_length(x['sim'][1], 0) < maxAlphaLength)
)